# 46 · Configuration-driven transformations

**Scenario:** Apply a validated metadata configuration to source columns without evaluating arbitrary Python or SQL.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **dataframe**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Step 2 — Define a constrained configuration

The configuration contains metadata, not replacement records. Allowlist columns and operations.


In [ ]:
config={'group_by':['pickup_zip'],'metrics':[{'column':'fare','operation':'sum','alias':'fare_total'},
    {'column':'distance','operation':'avg','alias':'mean_distance'}]}
allowed_operations={'sum':F.sum,'avg':F.avg,'min':F.min,'max':F.max}
def configured_aggregate(frame,spec):
    groups=spec['group_by']
    assert groups and all(c in frame.columns for c in groups)
    expressions=[]
    for metric in spec['metrics']:
        assert metric['column'] in frame.columns
        assert metric['operation'] in allowed_operations
        assert metric['alias'].replace('_','').isalnum()
        expressions.append(allowed_operations[metric['operation']](F.col(metric['column'])).alias(metric['alias']))
    return frame.groupBy(*groups).agg(*expressions)
result=configured_aggregate(df,config)
result.show(10)


**Expected check:** The metadata produces an origin-level aggregate from the asset.


## Step 3 — Fail invalid configurations before execution

Do not use eval/exec for transformation definitions supplied by users or metadata tables.


In [ ]:
invalid={'group_by':['pickup_zip'],'metrics':[{'column':'fare','operation':'shell','alias':'x'}]}
try:
    configured_aggregate(df,invalid)
except AssertionError:
    print('Invalid operation rejected')
else:
    raise AssertionError('Unsafe configuration accepted')
assert result.count()==df.select('pickup_zip').distinct().count()


**Expected check:** Unsupported operations fail validation before an aggregation runs.


## Practice

Store approved configurations in a governed Delta metadata table and validate required fields and duplicate aliases.

## Solution guide

Require a versioned schema and unique output aliases. Treat metadata changes like code changes; arbitrary SQL strings remain executable code even when stored in a table.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
